# SEIS 606: Vibe Coding
## Homework 2, Image Generation for App Specs
Dante Razo, razo3843@stthomas.edu, FA26

I've been using this GPU-accelerated notebook template since I first started at UST. It's something that I carry from class to class.

## GPU-Accelerated Environment Configuration

In [13]:
import torch

# validate CUDA setup
print("Torch CUDA Available? ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Torch CUDA Version:", torch.version.cuda)
    print("Torch cuDNN Version:", torch.backends.cudnn.version())

    # print GPU information
    for i in range(torch.cuda.device_count()):
        print(f"\nGPU {i}:", torch.cuda.get_device_name(device=i))

    # check NVIDIA driver
    !echo && nvidia-smi

# set device type
device: str = "cuda" if torch.cuda.is_available() else "cpu"

Torch CUDA Available?  True
Torch CUDA Version: 13.0
Torch cuDNN Version: 92400

GPU 0: NVIDIA GeForce RTX 5090

Fri Sep 25 00:30:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 615.71.08              KMD Version: 616.92        CUDA UMD Version: 13.4     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        On  |   00000000:0A:00.0  On |                  N/A |
|  0%   42C    P0             59W /  460W |   20023MiB /  32607MiB |      1%      Default |
|                          

In [14]:
import gc


def free_vram() -> None:
    gc.collect()
    torch.cuda.empty_cache()


# free now, though it should be empty with a fresh kernel
free_vram()

In [15]:
import os
from pathlib import Path

# create cache location
hf_home: Path = Path("/cache/huggingface")
hf_home.mkdir(parents=True, exist_ok=True)

# set environment variables for huggingface / transformers
os.environ["HF_HOME"] = str(object=hf_home)

In [16]:
# validate environment variables with assertions
assert hf_home.exists()
assert os.environ["HF_HOME"] == str(object=hf_home)

In [17]:
from dotenv import load_dotenv

# load environment, including HF token
load_dotenv()

False

## Loading the Image Generation Model
I had AI generate a list of models to try given my hardware, and I created the following struct to easily switch between them.

In [18]:
# define model structs for easy switching
from dataclasses import dataclass
from enum import Enum

from torch import bfloat16, dtype, float16


@dataclass
class Text2ImageModel:
    name: str
    torch_dtype: dtype = float16
    variant: str | None = ""
    use_safetensors: bool = True


class Models(Enum):
    FLUX2_KLEIN_4B = Text2ImageModel(name="black-forest-labs/FLUX.2-klein-4B")
    QWEN_IMAGE_2_1 = Text2ImageModel(name="Qwen/Qwen-Image-2.1", torch_dtype=bfloat16)
    SDXL_BASE = Text2ImageModel(name="stabilityai/stable-diffusion-xl-base-1.0")
    PLAYGROUND_V2_5 = Text2ImageModel(name="playgroundai/playground-v2.5-1024px-aesthetic")
    AURAFLOW = Text2ImageModel(name="fal/AuraFlow")
    PIXART_SIGMA = Text2ImageModel(name="PixArt-alpha/PixArt-Sigma-XL-2-1024-MS")
    SANA_1600M = Text2ImageModel(name="Efficient-Large-Model/Sana_1600M_1024px_diffusers")

In [19]:
# select text-to-image model
model: Text2ImageModel = Models.QWEN_IMAGE_2_1.value

In [20]:
from diffusers.pipelines.auto_pipeline import AutoPipelineForText2Image

# define pipeline object
pipeline: AutoPipelineForText2Image = AutoPipelineForText2Image.from_pretrained(
    pretrained_model_or_path=model.name,
    torch_dtype=model.torch_dtype,
    use_safetensors=model.use_safetensors,
    **({"variant": model.variant} if model.variant else {}),
).to(device)

/home/dante/code/coursework/seis-606-vibe/.venv/lib/python3.14/site-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


model_index.json:   0%|          | 0.00/447 [00:00<?, ?B/s]

ValueError: AutoPipeline can't find a pipeline linked to QwenImage21Pipeline for None

In [ ]:
# wrapper function for generation + persisting to disk
def generate_image(prompt: str, save_path: str = "app-mockup.png") -> None:
    image = pipeline(prompt=prompt).images[0]  # pyright: ignore[reportCallIssue]
    image.save(save_path)

In [ ]:
generate_image(prompt="A clean UI mockup for a homelab dashboard")

Guidance scale 4.0 is ignored for step-wise distilled models.
In file included from /home/dante/.pyenv/versions/3.14.7/include/python3.14/Python.h:14,
                 from /home/dante/code/coursework/seis-606-vibe/.venv/lib/python3.14/site-packages/triton/backends/nvidia/driver.c:9:
/home/dante/.pyenv/versions/3.14.7/include/python3.14/pyconfig.h:2031:9: warning: ‘_POSIX_C_SOURCE’ redefined
 2031 | #define _POSIX_C_SOURCE 200809L
      |         ^~~~~~~~~~~~~~~
In file included from /usr/include/x86_64-linux-gnu/bits/libc-header-start.h:33,
                 from /usr/include/stdlib.h:26,
                 from /home/dante/code/coursework/seis-606-vibe/.venv/lib/python3.14/site-packages/triton/backends/nvidia/include/cuda.h:56,
                 from /home/dante/code/coursework/seis-606-vibe/.venv/lib/python3.14/site-packages/triton/backends/nvidia/driver.c:1:
/usr/include/features.h:319:10: note: this is the location of the previous definition
  319 | # define _POSIX_C_SOURCE        202

  0%|          | 0/50 [00:00<?, ?it/s]